# Week 3 — Tensor & Pipeline Parallelism

When a model no longer fits in the memory of a single GPU even after data parallelism, the model itself must be **split across devices**. There are two principal strategies:

1. **Tensor Parallelism (Megatron-LM)** — partition the weight matrices of each layer across devices, so every device participates in every layer.
2. **Pipeline Parallelism (GPipe / 1F1B)** — assign whole layers to different devices and stream microbatches through them like an assembly line.

This notebook derives the communication cost and memory behaviour of each, then implements both on a toy Transformer.

## Learning objectives

1. Decompose $Y = XA$ along the **column** and **row** of $A$, and identify which decomposition pairs with which.
2. Derive the **pipeline bubble fraction** $\rho_{\text{bubble}} = (p-1)/m$ and prove the **1F1B** schedule's memory advantage over GPipe.
3. Build a manual 1F1B scheduler with two virtual stages.


## 1. Column-Parallel Decomposition

Write the output projection as $Y = X A$ where $A \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$. Partition $A$ along its **output dimension**:

$$
A \;=\; [\,A_1\;|\;A_2\;|\;\cdots\;|\;A_p\,], \qquad A_i \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}/p}.
$$

Each rank computes $Y_i = X A_i$. The full output is the concatenation $Y = [Y_1, \dots, Y_p]$.

* **Forward communication:** zero — $X$ is already replicated.
* **Backward communication:** one AllReduce for $\partial L / \partial X = \sum_i (\partial L / \partial Y_i) A_i^{\top}$.


## 2. Row-Parallel Decomposition

Partition $A$ along its **input dimension**:

$$
A \;=\; \begin{bmatrix} A_1 \\ A_2 \\ \vdots \\ A_p \end{bmatrix}, \qquad A_i \in \mathbb{R}^{(d_{\text{in}}/p) \times d_{\text{out}}}.
$$

If the input is also sharded as $X = [X_1, X_2, \dots, X_p]$ along the matching axis, then each rank computes $Y_i = X_i A_i$ and the global output is the **sum**

$$
Y \;=\; \sum_{i=1}^{p} X_i A_i,
$$

realised by an AllReduce.

### The Megatron-LM MLP pattern

A Transformer MLP is two linears with a non-linearity in between:

$$
Y \;=\; \mathrm{GeLU}(X W_1) W_2.
$$

By making $W_1$ **column-parallel** and $W_2$ **row-parallel**, the intermediate $\mathrm{GeLU}(X W_1)$ is already sharded along the contraction axis of $W_2$, and only **one AllReduce per MLP** is needed — at the output.


In [ ]:
# %% Column-parallel + row-parallel MLP
import torch
import torch.nn as nn

from hpcllmforge.parallelism.tensor.column_parallel import ColumnParallelLinear
from hpcllmforge.parallelism.tensor.row_parallel import RowParallelLinear

class TensorParallelMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int, tp_world_size: int, tp_rank: int):
        super().__init__()
        self.fc1 = ColumnParallelLinear(d_model, d_ff,
                                        tp_world_size=tp_world_size, tp_rank=tp_rank)
        self.fc2 = RowParallelLinear(d_ff, d_model,
                                     tp_world_size=tp_world_size, tp_rank=tp_rank,
                                     input_is_parallel=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(torch.nn.functional.gelu(self.fc1(x)))


## 3. Pipeline Parallelism: The Bubble

Assign the $L$ layers of the model to $p$ devices in $p$ contiguous blocks. A single forward-then-backward pass over a batch yields the following schedule (rows = devices, columns = time):

```
Stage 0:  F F F F . . . . B B B B
Stage 1:  . F F F F . . B B B B .
Stage 2:  . . F F F F B B B B . .
Stage 3:  . . . F F F F B B B . .
```

This **GPipe** schedule has

* **Bubble fraction** $\rho_{\text{bubble}} = (p-1)/m$ where $m$ is the microbatch count.
* **Activation memory** $\propto m$ — every microbatch's activations must be kept alive until its backward.

### 1F1B (One-Forward-One-Backward)

Once the pipeline is full, alternate one forward and one backward microbatch. The bubble fraction is the same, but **at most $p$ activations are alive per stage at once** — independent of $m$. This is the schedule used in production by Megatron-LM and DeepSpeed.


In [ ]:
# %% Two-stage 1F1B on virtual devices (CPU/GPU split for illustration)
import torch
import torch.nn as nn

from hpcllmforge.parallelism.pipeline.one_forward_one_backward import (
    OneForwardOneBackwardScheduler, PipelineStage,
)

stage_0 = PipelineStage(module=nn.Linear(1024, 1024).cuda(0), device='cuda:0', stage_id=0)
stage_1 = PipelineStage(module=nn.Linear(1024, 1024).cuda(1), device='cuda:1', stage_id=1)

scheduler = OneForwardOneBackwardScheduler([stage_0, stage_1], num_microbatches=8)
print(f"Bubble fraction: {scheduler.bubble_fraction:.2%}")


## 4. Combining Tensor + Pipeline + Data Parallelism

The frontier-scale recipe is **3D parallelism**: tensor parallelism inside each node, pipeline parallelism across nodes, data parallelism on top.

* TP degree $t$ — bounded by NVLink bandwidth (typically $t \le 8$ within a node).
* PP degree $p$ — bounded by the number of nodes.
* DP degree $d$ — fills the rest: total GPUs $= t \cdot p \cdot d$.

Memory per GPU scales as $O(1 / (t \cdot p))$; compute scales as $O(t \cdot p \cdot d)$.


## 5. Exercises

1. Implement column- and row-parallel linear layers on a single GPU using **virtual ranks** (manually slicing tensors). Verify mathematical equivalence to a plain `nn.Linear`.
2. Empirically measure the pipeline bubble for $p \in \{2, 4\}$ and $m \in \{1, 4, 16, 64\}$. Plot $\rho_{\text{bubble}}$ vs $m$ and compare to the theoretical curve.
3. Derive the communication cost of an AllReduce inside a tensor-parallel group vs. the AllReduce-over-DP-replicas. For what cluster topologies is each strategy preferable?
